# E1 — IPAD R01 Video Swin Tiny

Bu notebook, frozen pretrained Video Swin Tiny özellikleriyle E1 memory bank baseline’ını çalıştırır. **Run All kullanmayın:** önce smoke cell’ini çalıştırıp ardından `RUN_FULL_E1=True` değerini kullanıcı olarak değiştirin. E0 run klasörleri okunmaz veya değiştirilmez.

In [ ]:
PROJECT_ROOT = '/content/bitirme-ipad-r01'
DATASET_ROOT = '/content/ipad_data/R01'
PERSIST_ROOT = '/content/drive/MyDrive/bitirme-ipad-r01'
CACHE_DIR = f'{PERSIST_ROOT}/models/E1_video_swin16_cache'
OUTPUT_DIR = f'{PERSIST_ROOT}/models'
REPORTS_DIR = f'{PERSIST_ROOT}/outputs/reports'
BATCH_SIZE = 4  # OOM olursa 2 veya 1; yeterli VRAM varsa 8 deneyin
RUN_FULL_E1 = False  # Full E1'i bilerek kapalı tutar


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, sys, subprocess
from pathlib import Path
assert Path(PROJECT_ROOT).is_dir(), PROJECT_ROOT
assert Path(DATASET_ROOT).is_dir(), DATASET_ROOT
os.chdir(PROJECT_ROOT)
sys.path.insert(0, PROJECT_ROOT)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
import torch
print('torch', torch.__version__, 'CUDA', torch.cuda.is_available())
assert torch.cuda.is_available(), 'Select a Colab GPU runtime (T4 recommended)'
print(torch.cuda.get_device_name(0), round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1), 'GiB')


In [ ]:
from pathlib import Path
import pandas as pd
manifest_path = Path('data/processed/r01_manifest.csv')
if not manifest_path.exists() or not pd.read_csv(manifest_path).path.map(lambda p: Path(p).is_file()).all():
    subprocess.run([sys.executable, 'scripts/prepare_r01.py', '--root', DATASET_ROOT, '--output', str(manifest_path)], check=True)
manifest = pd.read_csv(manifest_path)
print(manifest.groupby(['split', 'label']).size())
print('sequence counts:', manifest.groupby('split').sequence.nunique().to_dict())


In [ ]:
# Smoke: one train, one validation, and one test sequence; no full E1 run.
subprocess.run([sys.executable, '-m', 'scripts.smoke_test_video', '--batch-size', '1'], check=True)


## Full E1 run

Smoke test PASS olduktan sonra `RUN_FULL_E1=True` yapıp bu hücreyi çalıştırın. Cache metadata uymazsa eğitim feature extraction’ı durur; yanlış cache kullanılmaz. Embedding cache `models/E1_video_swin16_cache/` altında kalır ve runtime kapanıp açıldığında tekrar kullanılabilir.

In [ ]:
if RUN_FULL_E1:
    subprocess.run([sys.executable, '-m', 'src.training.train_video_baseline', '--config', 'configs/video_swin.yaml', '--batch-size', str(BATCH_SIZE), '--cache-dir', CACHE_DIR, '--output-dir', OUTPUT_DIR], check=True)
else:
    print('Full E1 kapalı. RUN_FULL_E1=True yapın ve bu hücreyi ayrıca çalıştırın.')


In [ ]:
# Run evaluation only after the full training/feature-extraction command creates an E1 run.
if RUN_FULL_E1:
    run_dirs = sorted(Path(OUTPUT_DIR).glob('E1_video_swin16_*'), key=lambda p: p.stat().st_mtime, reverse=True)
    run_dir = next((p for p in run_dirs if (p / 'config.json').exists()), None)
    assert run_dir is not None, 'No E1 run found'
    subprocess.run([sys.executable, '-m', 'src.evaluation.evaluate_video', '--run-dir', str(run_dir), '--batch-size', str(BATCH_SIZE), '--cache-dir', CACHE_DIR, '--reports-dir', REPORTS_DIR], check=True)
    print('Evaluated:', run_dir)
else:
    print('Evaluation is not run because full E1 is disabled.')
